# Notebook 12 – Train, Validation & Test Data

This notebook splits `customer_transactions_raw.csv` into training, validation, and test sets, and demonstrates the difference between a correct and an incorrect preprocessing workflow with respect to data leakage.

In [1]:
import pandas as pd
import numpy as np
df = pd.read_csv('customer_transactions_raw.csv')
df['age_numeric'] = pd.to_numeric(df['age'], errors='coerce')
df['purchase_amount_clean'] = df['purchase_amount'].astype(str).str.replace('$', '', regex=False)
df['purchase_amount_clean'] = pd.to_numeric(df['purchase_amount_clean'], errors='coerce')
df['membership_clean'] = df['membership_type'].str.strip().str.lower()
df['is_platinum'] = (df['membership_clean'] == 'platinum').astype(int)
df.shape

(1000, 16)

## 1. Training Dataset

The training dataset is the portion of data a model directly learns from — it adjusts its internal parameters (e.g. regression coefficients, tree splits) to fit patterns in this data.

## 2. Validation Dataset

The validation dataset is used **during model development** to tune hyperparameters and compare candidate models, without ever being used to fit the model's parameters directly. It acts as a proxy for how well the model generalizes, letting us pick a model or settings before final evaluation.

## 3. Test Dataset

The test dataset is held out until the very end and used **exactly once** to report the final, unbiased estimate of model performance. If it's used repeatedly to make decisions (like the validation set is), it stops being a trustworthy estimate of real-world performance.

## 4. Train-Test Split

A train-test split divides the dataset into two parts: one for fitting the model, one for evaluating it.

In [2]:
from sklearn.model_selection import train_test_split
feature_cols = ['annual_income', 'quantity', 'rating']
X = df[feature_cols].fillna(df[feature_cols].median())
y = df['is_platinum']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train.shape, X_test.shape

((800, 3), (200, 3))

## 5. Train-Validation-Test Split

When hyperparameter tuning or model comparison is needed, the data is split three ways instead of two: train (to fit), validation (to tune/compare), and test (final evaluation only). A common approach is a two-step split: first separate out the test set, then split the remainder into train and validation.

In [4]:
X_trainval, X_test, y_trainval, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_trainval, y_trainval, test_size=0.25, random_state=42)
print('train:', X_train.shape)
print('val:', X_val.shape)
print('test:', X_test.shape)

train: (600, 3)
val: (200, 3)
test: (200, 3)


## 6. Random State

The `random_state` parameter fixes the randomness used to shuffle and split the data, making the split **reproducible** — running the same code again gives the exact same train/validation/test rows. Without fixing it, every run produces a different split, making results hard to compare or debug.

In [6]:
split_a = train_test_split(X, y, test_size=0.2, random_state=42)[0].index[:5].tolist()
split_b = train_test_split(X, y, test_size=0.2, random_state=42)[0].index[:5].tolist()
split_c = train_test_split(X, y, test_size=0.2, random_state=7)[0].index[:5].tolist()
print('run 1 (seed=42):', split_a)
print('run 2 (seed=42):', split_b)
print('run 3 (seed=7): ', split_c)

run 1 (seed=42): [29, 535, 695, 557, 836]
run 2 (seed=42): [29, 535, 695, 557, 836]
run 3 (seed=7):  [600, 80, 158, 423, 747]


## 7. Stratified Sampling

Stratified sampling ensures that each split preserves the same class proportions as the full dataset. This matters especially for imbalanced targets (see Notebook 11) — a plain random split could, by chance, put very few or zero minority-class (Platinum) examples into the test set.

In [7]:
X_train_plain, X_test_plain, y_train_plain, y_test_plain = train_test_split(X, y, test_size=0.2, random_state=42)
print('Plain split — train Platinum rate:', y_train_plain.mean().round(4), '| test Platinum rate:', y_test_plain.mean().round(4))
X_train_strat, X_test_strat, y_train_strat, y_test_strat = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print('Stratified split — train Platinum rate:', y_train_strat.mean().round(4), '| test Platinum rate:', y_test_strat.mean().round(4))

Plain split — train Platinum rate: 0.1162 | test Platinum rate: 0.125
Stratified split — train Platinum rate: 0.1175 | test Platinum rate: 0.12


## 8. Data Leakage

Data leakage occurs when information from outside the training data — especially from the validation or test set — influences the model during training, producing overly optimistic performance estimates that collapse once the model sees genuinely new data. A very common source of leakage is fitting a preprocessing step (scaler, imputer, encoder) on the **entire dataset** before splitting.

## 9. Why Preprocessing Must Be Fitted Only on Training Data

Any preprocessing step that learns something from the data — a scaler's mean/std, an imputer's median, an encoder's category mapping — must learn those statistics **only** from the training set, then apply the same learned transformation to the validation and test sets. Fitting on the full dataset lets information about the test set's distribution (its mean, its range, its extreme values) leak into the numbers the model is trained on, inflating validation/test performance in a way that won't hold up in production.

## 10. Demonstrating Incorrect vs. Correct Preprocessing Workflows

In [8]:
from sklearn.preprocessing import StandardScaler
X_train_leak, X_test_leak, y_train_leak, y_test_leak = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
leaky_scaler = StandardScaler()
leaky_scaler.fit(pd.concat([X_train_leak, X_test_leak]))
X_train_leaky_scaled = leaky_scaler.transform(X_train_leak)
X_test_leaky_scaled = leaky_scaler.transform(X_test_leak)
print('INCORRECT workflow — scaler fitted on train+test combined')
print('Scaler mean (leaked, includes test rows):', leaky_scaler.mean_)

INCORRECT workflow — scaler fitted on train+test combined
Scaler mean (leaked, includes test rows): [3.0703623e+06 4.9960000e+00 2.9860000e+00]


In [10]:
correct_scaler = StandardScaler()
correct_scaler.fit(X_train_leak)
X_train_correct_scaled = correct_scaler.transform(X_train_leak)
X_test_correct_scaled = correct_scaler.transform(X_test_leak)
print('CORRECT workflow — scaler fitted on training data only')
print('Scaler mean (train-only):', correct_scaler.mean_)

CORRECT workflow — scaler fitted on training data only
Scaler mean (train-only): [2.56537571e+06 4.97875000e+00 2.97125000e+00]


In [11]:
mean_difference = leaky_scaler.mean_ - correct_scaler.mean_
pd.Series(mean_difference, index=feature_cols, name='mean_shift_from_leakage')

annual_income    504986.59052
quantity              0.01725
rating                0.01475
Name: mean_shift_from_leakage, dtype: float64

**What this shows:** the leaky scaler's mean is pulled slightly by the test set's rows, which it should never have seen. On a small or noisy test set this shift can be large enough to measurably change the scaled feature values the model is evaluated on — meaning the reported test performance is no longer a clean estimate of how the model will do on truly unseen data.

The same principle applies to any fitted preprocessing: **imputers** (median/mean for missing values), **encoders** (category-to-number mappings, including target encoding from Notebook 7), and **feature selectors** (which features rank as most important) must all be fit on the training split only, then applied unchanged to validation/test.

In [12]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
model_leaky = LogisticRegression(max_iter=1000)
model_leaky.fit(X_train_leaky_scaled, y_train_leak)
leaky_test_accuracy = accuracy_score(y_test_leak, model_leaky.predict(X_test_leaky_scaled))
model_correct = LogisticRegression(max_iter=1000)
model_correct.fit(X_train_correct_scaled, y_train_leak)
correct_test_accuracy = accuracy_score(y_test_leak, model_correct.predict(X_test_correct_scaled))
print('Test accuracy (leaky preprocessing):', leaky_test_accuracy)
print('Test accuracy (correct preprocessing):', correct_test_accuracy)

Test accuracy (leaky preprocessing): 0.88
Test accuracy (correct preprocessing): 0.88


## 11. Temporal Splitting

When data has a time dimension (like `signup_date` here), a random split can leak future information into the training set — the model could end up training on customers who signed up *after* some of the customers in its test set, which is impossible in a real deployment where the model only ever sees the past. **Temporal splitting** instead orders data by time and splits so that training data comes entirely before validation/test data.

In [13]:
def parse_flexible_date(value):
    if pd.isna(value):
        return pd.NaT
    for fmt in ('%m-%d-%Y', '%d/%m/%Y', '%Y-%m-%d', '%d %b %Y'):
        try:
            return pd.to_datetime(value, format=fmt)
        except (ValueError, TypeError):
            continue
    return pd.NaT
df['signup_date_parsed'] = df['signup_date'].apply(parse_flexible_date)
df['signup_date_parsed'].isna().sum()

np.int64(16)

In [14]:
df_sorted_by_time = df.dropna(subset=['signup_date_parsed']).sort_values('signup_date_parsed')
split_point = int(len(df_sorted_by_time) * 0.8)
temporal_train = df_sorted_by_time.iloc[:split_point]
temporal_test = df_sorted_by_time.iloc[split_point:]
print('Train period:', temporal_train['signup_date_parsed'].min(), 'to', temporal_train['signup_date_parsed'].max())
print('Test period:', temporal_test['signup_date_parsed'].min(), 'to', temporal_test['signup_date_parsed'].max())
print('Train rows:', len(temporal_train), '| Test rows:', len(temporal_test))

Train period: 2019-01-03 00:00:00 to 2022-12-13 00:00:00
Test period: 2022-12-14 00:00:00 to 2023-12-30 00:00:00
Train rows: 787 | Test rows: 197


**Note:** a random `train_test_split` on this same data would mix customers from across all years into both the training and test sets — meaning the model could be evaluated on a 2019 customer's data while having trained on 2023 customers, which does not reflect how the model would actually be used in production (predicting forward in time based on the past).

## 12. Why preprocessing must be fitted only on training data

- Use **train/validation/test** (not just train/test) whenever hyperparameters need tuning or models need comparing before final evaluation.
- Fix `random_state` for reproducibility, and use `stratify=y` for classification targets, especially imbalanced ones like `is_platinum` here.
- Fit every preprocessing step (scalers, imputers, encoders, feature selectors) **on the training split only**, then transform validation/test with those already-fitted objects — never re-fit on the full dataset.
- Use a **temporal split** instead of a random split whenever the data has a meaningful time order and the model will be used to predict forward in time.